In [24]:
import pandas as pd
import numpy as np
from alpha_vantage.timeseries import TimeSeries
from datetime import datetime

class ScriptData:
    def __init__(self):
        self.api_key = 'YOUR_API_KEY'
        self.ts = TimeSeries(key=self.api_key, output_format='pandas')

    def fetch_intraday_data(self, symbol, interval):
        data, meta_data = self.ts.get_intraday(symbol=symbol, interval=interval, outputsize='full')
        self.raw_data = data
        self.symbol = symbol

    def convert_intraday_data(self):
        self.df = self.raw_data.reset_index()
        self.df.columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
        self.df['timestamp'] = pd.to_datetime(self.df['timestamp'])
        self.df.set_index('timestamp', inplace=True)

    def __getitem__(self, key):
        return self.df[key]

    def __setitem__(self, key, value):
        self.df[key] = value

    def __contains__(self, key):
        return key in self.df.columns

def indicator1(df, timeperiod):
    indicator = pd.DataFrame()
    indicator['timestamp'] = df.index
    indicator['indicator'] = df['close'].rolling(timeperiod).mean()
    indicator.dropna(inplace=True)
    indicator.reset_index(drop=True, inplace=True)
    return indicator

class Strategy:
    def __init__(self, symbol, interval):
        self.script_data = ScriptData()
        self.symbol = symbol
        self.interval = interval

    def get_script_data(self):
        self.script_data.fetch_intraday_data(self.symbol, self.interval)
        self.script_data.convert_intraday_data()

    def get_signals(self):
        self.indicator_data = indicator1(self.script_data.df, 5)
        self.signals = pd.DataFrame()
        self.signals['timestamp'] = self.script_data.df.index
        self.signals['signal'] = 'NO_SIGNAL'
        for i in range(1, len(self.indicator_data)):
            if self.indicator_data.iloc[i - 1]['indicator'] < self.script_data.df.iloc[i - 1]['close'] and \
                    self.indicator_data.iloc[i]['indicator'] >= self.script_data.df.iloc[i]['close']:
                self.signals.loc[i, 'signal'] = 'BUY'
            elif self.indicator_data.iloc[i - 1]['indicator'] > self.script_data.df.iloc[i - 1]['close'] and \
                    self.indicator_data.iloc[i]['indicator'] <= self.script_data.df.iloc[i]['close']:
                self.signals.loc[i, 'signal'] = 'SELL'

symbols = ['IBM', 'GOOGL', 'AAPL', 'NVDA']
interval = '60min'
strategies = []
for symbol in symbols:
    strategy = Strategy(symbol, interval)
    strategy.get_script_data()
    strategy.get_signals()
    strategies.append(strategy)
